# Selenium

In [ ]:
import csv
import json
import os
import time
import random
from datetime import datetime, timedelta
import urllib.parse
import re

from selenium import webdriver
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException, TimeoutException
from selenium.webdriver import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager

# ================= Settings =================

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/121.0.0.0 Safari/537.36"
)

OUTPUT_DIRECTORY = "output"
TWEET_DATA_DIRECTORY = os.path.join(OUTPUT_DIRECTORY, "tweet_data")
POSTS_FILE_TEMPLATE = os.path.join(OUTPUT_DIRECTORY, "posts_{idx}.txt")
COOKIES_FILE = "cookies.json"
MAX_TWEETS = 10_000  # Увеличим цель для запаса
CSV_OUTPUT = os.path.join(OUTPUT_DIRECTORY, "volleyball_tweets_11_04.csv")

# ============================================

def create_directory(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def configure_chrome_options():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--log-level=3")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument(f"user-agent={USER_AGENT}")
    return options

def initialize_driver():
#     options = configure_chrome_options()
#     service = Service(ChromeDriverManager().install())
#     driver = webdriver.Chrome(service=service, options=options)
#     driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    service = Service()
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(service=service, options=options)
    
    return driver

def load_cookies(driver):
    if not os.path.exists(COOKIES_FILE):
        print("⚠️ cookies.json не найден. Продолжаем без авторизации.")
        return False

    try:
        driver.get("https://x.com/")
        time.sleep(3)
        
        with open(COOKIES_FILE, "r", encoding="utf-8") as f:
            cookies = json.load(f)
        
        for cookie in cookies:
            cookie.pop("sameSite", None)
            cookie.pop("storeId", None)
            cookie.pop("session", None)
            try:
                driver.add_cookie(cookie)
            except:
                pass
        
        driver.refresh()
        time.sleep(5)
        return True
    except Exception as e:
        print(f"Ошибка загрузки cookies: {e}")
        return False

def is_english_text(text):
    """Проверяет, содержит ли текст преимущественно английские символы"""
    if not text:
        return False
    
    # Подсчитываем английские буквы и пробелы vs другие символы
    english_chars = sum(1 for c in text if c.isalpha() and ord(c) < 128)
    other_chars = sum(1 for c in text if c.isalpha() and ord(c) >= 128)
    
    # Если больше 70% символов - английские, считаем текст английским
    total_alpha = english_chars + other_chars
    if total_alpha == 0:
        return False
    
    return (english_chars / total_alpha) > 0.7

def build_volleyball_queries():
    """
    Создает расширенный список запросов по волейболу с фокусом на английский язык
    """
    # Базовые термины
    base_terms = [
        "volleyball",
        "beach volleyball",
        "indoor volleyball",
    ]
    
    # Команды и лиги
    teams_leagues = [
        "FIVB",
        "Volleyball Nations League",
        "VNL",
        "World Championship volleyball",
        "Olympic volleyball",
        "NCAA volleyball",
        "CEV Champions League",
    ]
    
    # Игроки (современные)
    players = [
        "Wilfredo Leon",
        "Micah Christenson",
        "Yuji Nishida",
        "Earvin Ngapeth",
        "Ivan Zaytsev",
        "Bartosz Kurek",
        "Simone Giannelli",
        "Thales Hoss",
        "Wallace Souza",
        "Yoandy Leal",
    ]
    
    # Термины игры
    game_terms = [
        "volleyball spike",
        "volleyball block",
        "volleyball serve",
        "volleyball setter",
        "volleyball libero",
        "volleyball match",
        "volleyball game today",
        "volleyball highlights",
    ]
    
    # Эмоциональные запросы (для сбора данных с разными эмоциями)
    emotional_queries = [
        "I love volleyball",
        "volleyball victory",
        "won volleyball match",
        "lost volleyball game",
        "amazing volleyball play",
        "terrible volleyball call",
        "volleyball disappointment",
        "volleyball celebration",
        "volleyball championship win",
        "volleyball heartbreak",
    ]
    
    # Объединяем все запросы
    all_queries = base_terms + teams_leagues + players + game_terms + emotional_queries
    
    # Создаем варианты с хэштегами
    hashtag_queries = [f"#{term.replace(' ', '')}" for term in base_terms + teams_leagues[:3]]
    
    return all_queries + hashtag_queries

def generate_date_ranges():
    """
    Генерирует диапазоны дат за последние 2 года по месяцам
    """
    date_ranges = []
    end_date = datetime.now()
    start_date = end_date - timedelta(days=730)  # 2 года назад
    
    current = start_date
    while current < end_date:
        month_end = current + timedelta(days=30)
        if month_end > end_date:
            month_end = end_date
        
        date_str = f"since:{current.strftime('%Y-%m-%d')} until:{month_end.strftime('%Y-%m-%d')}"
        date_ranges.append(date_str)
        
        current = month_end + timedelta(days=1)
    
    return date_ranges

def extract_tweet_text(tweet_element):
    """Улучшенное извлечение текста с проверкой языка"""
    
    text_selectors = [
        '[data-testid="tweetText"]',
        'div[lang]',
        'div[dir="auto"]',
        'span.css-1jxf684',
    ]
    
    for selector in text_selectors:
        try:
            elements = tweet_element.find_elements(By.CSS_SELECTOR, selector)
            for element in elements:
                text = element.text
                if text and len(text) > 15:  # Минимальная длина
                    # Проверяем, что это не реклама
                    if any(word in text.lower() for word in ['subscribe', 'sign up', 'download now', 'promoted']):
                        continue
                    
                    # Проверяем язык
                    if is_english_text(text):
                        return text
        except:
            continue
    
    return None

def scrape_tweets_for_query(driver, query, date_range, max_tweets=300):
    """
    Собирает твиты для конкретного запроса с датами
    """
    tweets = []
    seen_texts = set()
    scroll_count = 0
    no_new_tweets_count = 0
    max_no_new = 15
    
    # Формируем полный запрос
    full_query = f"{query} {date_range} lang:en -filter:retweets"
    encoded_query = urllib.parse.quote(full_query)
    search_url = f"https://x.com/search?q={encoded_query}&src=typed_query&f=live"
    
    print(f"\n  🔍 Запрос: {full_query}")
    driver.get(search_url)
    time.sleep(5)
    
    # Переключаемся на Latest
    try:
        latest_tab = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//a[@role='tab' and contains(@href, 'f=live')]"))
        )
        latest_tab.click()
        time.sleep(3)
    except:
        print("  ⚠️ Не удалось переключиться на Latest")
    
    last_height = driver.execute_script("return document.body.scrollHeight")
    
    while len(tweets) < max_tweets and no_new_tweets_count < max_no_new:
        scroll_count += 1
        
        # Скроллим
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(random.uniform(2, 4))
        
        # Собираем твиты
        tweet_elements = driver.find_elements(By.CSS_SELECTOR, '[data-testid="tweet"]')
        
        before_count = len(tweets)
        
        for tweet in tweet_elements:
            try:
                text = extract_tweet_text(tweet)
                if text and text not in seen_texts:
                    # Дополнительная проверка на волейбол в тексте
                    volleyball_keywords = ['volley', 'spike', 'block', 'serve', 'fivb', 'match point', 
                                         'setter', 'libero', 'beach volleyball', 'court']
                    
                    # Для общих запросов проверяем наличие ключевых слов
                    if query.lower() in ['volleyball', 'beach volleyball']:
                        if any(keyword in text.lower() for keyword in volleyball_keywords):
                            seen_texts.add(text)
                            tweets.append(text)
                    else:
                        # Для специфичных запросов добавляем все
                        seen_texts.add(text)
                        tweets.append(text)
            except StaleElementReferenceException:
                continue
        
        # Прогресс
        if len(tweets) > before_count:
            print(f"  📊 Прогресс: {len(tweets)}/{max_tweets} твитов", end='\r')
            no_new_tweets_count = 0
        else:
            no_new_tweets_count += 1
        
        # Проверка конца страницы
        new_height = driver.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            no_new_tweets_count += 1
        last_height = new_height
    
    print(f"\n  ✅ Собрано {len(tweets)} твитов")
    return tweets

def save_to_csv(tweets, filename):
    """Сохраняет твиты в CSV для удобной разметки"""
    with open(filename, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['id', 'text', 'sentiment'])  # sentiment оставляем пустым для разметки
        
        for i, tweet in enumerate(tweets, 1):
            # Очищаем текст от лишних пробелов и переносов
            clean_text = ' '.join(tweet.split())
            writer.writerow([i, clean_text, ''])


In [ ]:

def main():
    """Основная функция"""
    print("🚀 ЗАПУСК РАСШИРЕННОГО ПАРСЕРА ТВИТОВ О ВОЛЕЙБОЛЕ")
    print("🎯 Цель: собрать 1000+ английских твитов для разметки эмоций")
    
    # Создаем директории
    create_directory(OUTPUT_DIRECTORY)
    create_directory(TWEET_DATA_DIRECTORY)
    
    # Получаем запросы и даты
    queries = build_volleyball_queries()
    date_ranges = generate_date_ranges()
    
    print(f"📋 Запросов: {len(queries)}")
    print(f"📅 Диапазонов дат: {len(date_ranges)}")
    print(f"📊 Всего комбинаций: {len(queries) * len(date_ranges)}")
    
    # Инициализируем драйвер
    driver = initialize_driver()
    
    all_tweets = set()
    
    try:
        # Загружаем cookies
        load_cookies(driver)
        
        total_combinations = len(queries) * len(date_ranges)
        current_combination = 0
        
        # Перебираем все комбинации
        for query in queries:
            if len(all_tweets) >= MAX_TWEETS:
                break
                
            for date_range in date_ranges:
                if len(all_tweets) >= MAX_TWEETS:
                    break
                
                current_combination += 1
                print(f"\n{'='*60}")
                print(f"КОМБИНАЦИЯ {current_combination}/{total_combinations}")
                print(f"Всего собрано: {len(all_tweets)}/{MAX_TWEETS}")
                print(f"{'='*60}")
                
                # Для каждой комбинации собираем твиты
                tweets = scrape_tweets_for_query(driver, query, date_range, max_tweets=100)
                
                # Добавляем новые твиты
                before = len(all_tweets)
                all_tweets.update(tweets)
                new_tweets = len(all_tweets) - before
                
                print(f"✨ Новых уникальных твитов: {new_tweets}")
                
                # Сохраняем промежуточные результаты
                if len(all_tweets) % 200 < new_tweets or len(all_tweets) >= MAX_TWEETS:
                    # Сохраняем в CSV
                    save_to_csv(list(all_tweets), CSV_OUTPUT)
                    
                    # Сохраняем текстовый файл для просмотра
                    txt_file = POSTS_FILE_TEMPLATE.format(idx=len(all_tweets))
                    with open(txt_file, 'w', encoding='utf-8') as f:
                        tweets_list = list(all_tweets)
                        random.shuffle(tweets_list)  # Перемешиваем для разнообразия
                        for i, tweet in enumerate(tweets_list[:100], 1):
                            f.write(f"{i}. {tweet}\n\n")
                    
                    print(f"💾 Сохранено {len(all_tweets)} твитов в CSV и TXT")
                
                # Пауза между запросами
                time.sleep(random.randint(3, 7))
        
        # Финальное сохранение
        save_to_csv(list(all_tweets), CSV_OUTPUT)
        
        # Статистика
        print(f"\n{'='*60}")
        print(f"🎉 СБОР ЗАВЕРШЕН!")
        print(f"📊 Всего уникальных твитов: {len(all_tweets)}")
        print(f"💾 CSV файл для разметки: {CSV_OUTPUT}")
        
        # Примеры твитов
        print(f"\n📝 Примеры собранных твитов:")
        tweets_list = list(all_tweets)
        random.shuffle(tweets_list)
        for i, tweet in enumerate(tweets_list[:10], 1):
            print(f"{i}. {tweet[:100]}...")
        
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        import traceback
        traceback.print_exc()
        
    finally:
        driver.quit()



In [ ]:
if __name__ == "__main__":
    main()